## Loading and Displaying Images

# Lesson 1: Building Your Image Workbench

Welcome to the course! Over the next four units, we will build a complete panorama stitcher from scratch using Python. We will learn how to take multiple overlapping photos and blend them into one seamless wide-angle image.

Before we can perform any complex math or image stitching, our first step is to build a reliable workbench tool. We need a way to load an image, inspect its properties, and view it on our screen. This basic setup provides a dependable way to check our inputs before we run any advanced algorithms.

To do this, we will use **OpenCV** (imported as `cv2`), which is one of the most popular and powerful computer vision libraries in the world. While you will need to learn how to install OpenCV on your own personal computer, in the CodeSignal environment, OpenCV and all other necessary tools come pre-installed and ready to use!

Also, you will have a small **Image Gallery** to help you debug your work along the exercises!

---

## What is a Digital Image?

Before we dive deeper into the code, it is important to understand what a digital image actually is. To our eyes, it’s a photograph, but to a computer, it is a three-dimensional grid of numbers, also known as a **multidimensional array**.

* **Pixels:** The grid is made up of individual dots called pixels. Every pixel in the image contains numerical data that tells the computer what color to display.
* **Spatial Dimensions:** The first two dimensions of this grid represent the height and width of the image (the number of pixels stacked vertically and horizontally).
* **Channels:** The third dimension represents color channels.

---

## Understanding Color Channels

In digital imaging, colors are created by mixing primary colors of light. Most images use the **RGB** color model, which consists of three channels: Red, Green, and Blue.

Think of these channels as three separate grayscale layers. When you stack them on top of each other, they blend to create a full-color image. Each pixel in a standard 8-bit image has a value for each channel ranging from `0` (no intensity) to `255` (full intensity). For example:

* A pixel with values `(255, 0, 0)` in RGB is pure **Red**.
* A pixel with `(0, 0, 0)` is **Black** (all lights off).
* A pixel with `(255, 255, 255)` is **White** (all colors at maximum intensity).

> **The OpenCV Difference:** While the world usually uses RGB, **OpenCV reads images in BGR (Blue-Green-Red) order**. This means the first layer of data is Blue, the second is Green, and the third is Red. We must keep this in mind whenever we manipulate individual color values!

---

## Taking Input and Loading the Image

Let's start by setting up a way to tell our Python script which image we want to load. We can use Python's built-in `argparse` library to read command-line arguments. This allows us to pass a file path and other options directly into our program.

```python
import argparse
import cv2

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--out", default="")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

```

In this code, we expect the user to provide a path to the image. We also set up optional arguments for saving an output file (`--out`) and setting a maximum window size for our screen (`--preview-size`).

Next, we need to load the image into our program using `cv2.imread()`.

```python
    image = cv2.imread(args.path)
    
    if image is None:
        raise SystemExit(f"Could not read image: {args.path}")

```

Here, `cv2.imread()` attempts to read the file. If the file path is incorrect or the file is corrupted, OpenCV will not crash immediately; instead, it simply returns `None`. Therefore, it is very important to include a safety check. If the image is `None`, we stop the program and print a helpful error message so that we know exactly what went wrong.

---

## Inspecting Image Properties

To a computer, an image is not a picture; it is simply a massive grid of numbers known as an array. Let's add some code to inspect the mathematical properties of our newly loaded image.

```python
    print("shape:", image.shape)
    print("dtype:", image.dtype)
    print("min/max:", image.min(), image.max())
    print("note: OpenCV loads color images as BGR, not RGB")

```

Let's break down what these properties tell us:

* **`shape`:** This tells us the dimensions of the image. For a color image, it returns three numbers: **height, width, and color channels**.
* **`dtype`:** This is the data type. Typically, it is `uint8`, which stands for an **8-bit unsigned integer**.
* **`min` and `max`:** These are the lowest and highest pixel values in the image grid. Because we are using 8-bit integers, the lowest possible value is `0` (pure black) and the highest is `255` (pure white or full color).

If you were to run this code on a standard photo, you would see output that looks something like this:

```text
shape: (4000, 6000, 3)
dtype: uint8
min/max: 0 255
note: OpenCV loads color images as BGR, not RGB

```

Note the warning we printed at the end! By default, almost all computer graphics use Red-Green-Blue (RGB) order for colors. However, OpenCV has a famous quirk: it loads colors in **Blue-Green-Red (BGR)** order. It is very helpful to print this reminder so that we do not forget it later.

---

## Resizing for a Better Preview

If you look at the sample output above, the image has a height of 4,000 pixels and a width of 6,000 pixels. Modern photos are huge! If we try to show this image on a standard computer monitor pixel-for-pixel, it will be much larger than the screen.

To fix this, we will write a custom helper function to shrink the image specifically for our preview window.

```python
def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    
    if scale == 1.0:
        return image.copy()
        
    return cv2.resize(image, (int(width * scale), int(height * scale)))

```

Let's explain the math behind this step by step:

1. First, we examine the image shape to get the `height` and `width`.
2. Next, we determine which side is the longest using `max(height, width)`.
3. We divide our target `max_size` (which defaults to 900 pixels) by that longest edge to find our scale percentage. For example, if the image is 1800 pixels wide, $900 / 1800$ gives us a scale of $0.5$ (or $50\%$).
4. We use `min(1.0, ...)` to ensure that if the image is already smaller than 900 pixels, our scale remains at $1.0$. We do not want to stretch small images.
5. Finally, we use `cv2.resize()` to shrink the image to its new dimensions.

It is crucial to note that this resizing is **strictly for our display screen**. We want to keep the original, high-quality image array untouched for our actual stitching work later.

---

## Displaying and Saving the Image

Now that we have our original image loaded and a handy `resize_long_edge` function ready, let's complete our `main()` function by displaying the image to the user.

```python
    if args.out:
        cv2.imwrite(args.out, image)
        
    cv2.imshow("preview", resize_long_edge(image, max_size=args.preview_size))
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

```

Here is what these final few lines do:

1. First, we check if the user provided an `--out` argument. If they did, we use `cv2.imwrite()` to save a copy of the full-resolution image to their computer.
2. Next, we use `cv2.imshow()`. This command usually creates a pop-up window named `"preview"` and displays the result of our `resize_long_edge` function (or in the CodeSignal environment, renders it in the Image Gallery).
3. **Crucial Step:** We must call `cv2.waitKey(0)`. This tells Python to pause the program indefinitely (where `0` signifies forever) until the user presses a key on their keyboard. If we forget this line, the window will appear and disappear in a fraction of a second before we can even see it!
4. Finally, once the user presses a key, `cv2.destroyAllWindows()` safely closes the preview window and cleans up the program.

---

## Summary and Practice Prep

Excellent work! We have successfully built the foundation for our workbench:

* Learned how to use command-line arguments to pass in a file path.
* Used `cv2.imread()` to read the image safely.
* Explored how to inspect mathematical properties of the image array, such as its shape and data type.
* Created a custom mathematical helper function to resize the image.
* Used `cv2.imshow()` and `cv2.waitKey(0)` to safely display it.

You are now ready to put these concepts to work. In the upcoming interactive exercises, you will practice writing these loading, inspecting, and display functions yourself!

## Loading Your First Image Safely

Now that the lesson has walked you through the basics of loading images with OpenCV, it's time to set up the very first piece of your stitching workbench: a small script that accepts a command-line path and loads an image safely.

Open solution.py and follow the TODOs to build the script step by step:

    Import argparse and cv2 at the top of the file.
    Define a main() function that creates an ArgumentParser with a positional path argument, along with optional --out (default "") and --preview-size (an int, default 900).
    Inside main(), parse the arguments into args, then load the image with cv2.imread(args.path).
    If the loaded image is None, raise SystemExit with a helpful message.
    Finish with the standard block that calls main().

This little script is the launchpad for everything else in the workbench — get it solid, and the rest will come together smoothly.

```
# TODO: Import the argparse and cv2 libraries at the top of the file.


# TODO: Create a main() function that does the following:
#   1. Builds an Argument Parser.
#   2. Adds a positional argument called "path".
#   3. Adds an optional "--out" argument with default value "".
#   4. Adds an optional "--preview-size" argument of type int with default 900.
#   5. Parses the arguments into a variable called args.
#   6. Loads the image and stores it in a variable.
#   7. If the loaded image is None, raise SystemExit with a helpful message


# TODO: Add the standard "if __name__ == '__main__':" block that calls main().
```

Here is the complete implementation for `solution.py` following all the TODO steps to safely parse command-line arguments and load an image with OpenCV:

```python
# Import the argparse and cv2 libraries at the top of the file.
import argparse
import cv2


# Create a main() function that parses arguments and loads the image safely.
def main():
    # 1-4. Build an Argument Parser and add positional & optional arguments
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--out", default="")
    parser.add_argument("--preview-size", type=int, default=900)

    # 5. Parse the arguments into a variable called args
    args = parser.parse_args()

    # 6. Load the image and store it in a variable
    image = cv2.imread(args.path)

    # 7. If the loaded image is None, raise SystemExit with a helpful message
    if image is None:
        raise SystemExit(f"Could not read image: {args.path}")


# Add the standard "if __name__ == '__main__':" block that calls main().
if __name__ == "__main__":
    main()

```

## Inspecting Image Properties at a Glance

Nice work setting up the safe image loader in the last exercise! Now it is time to enable your workbench to provide specific information about the image it just loaded.

Immediately below the safety check, add four print statements so the script reports the image's properties:

    The image's shape, prefixed with "shape:".
    The image's data type, prefixed with "dtype:".
    The minimum and maximum pixel values, prefixed with "min/max:".
    A note starting with "note:" reminding the reader that OpenCV loads color images as BGR, rather than RGB.

Use image.shape, image.dtype, image.min(), and image.max() to obtain the necessary values.

Once complete, your workbench will provide a quick X-ray view of any image you load — a small step that will save you significant debugging time later.

```
import argparse
import cv2


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--out", default="")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = cv2.imread(args.path)
    if image is None:
        raise SystemExit(f"Could not read image: {args.path}")

    # TODO: Add four print statements that show the image's properties:
    #   1. The image's shape (use image.shape).
    #   2. The image's data type (use image.dtype).
    #   3. The min and max pixel values (use image.min() and image.max()).
    #   4. A note reminding the reader that OpenCV loads color images as BGR, not RGB.


if __name__ == "__main__":
    main()

```

Here is the complete code with the four print statements added to inspect the image properties:

```python
import argparse
import cv2


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--out", default="")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = cv2.imread(args.path)
    if image is None:
        raise SystemExit(f"Could not read image: {args.path}")

    # Add four print statements that show the image's properties
    print("shape:", image.shape)
    print("dtype:", image.dtype)
    print("min/max:", image.min(), image.max())
    print("note: OpenCV loads color images as BGR, not RGB")


if __name__ == "__main__":
    main()

```

## Shrinking Images for a Cleaner Preview

Skip to main content
Loading and Displaying Images

Nice work getting that quick "X-ray view" of your image up and running. Now, it's time to provide your workbench with a useful new tool: a helper function that shrinks images for preview purposes.

Large images can be slow to display and may overflow the screen, so you will write a small function called resize_long_edge that scales an image so that its longest side fits within a given size.

Inside solution.py, define resize_long_edge(image, max_size=900) so that it performs the following:

    Reads the height and width from image.shape[:2].
    Computes a scale factor as min(1.0, max_size / max(height, width)).
    If the scale equals 1.0, it returns image.copy() so that small images are left untouched.
    Otherwise, it returns cv2.resize(image, (int(width * scale), int(height * scale))).

The min(1.0, ...) logic is important: it ensures that small images are never upscaled, while only large ones are downscaled.

This helper will become a key building block once you start previewing and saving stitched panoramas later in the course.

```
import argparse
import cv2


# TODO: Define a helper function named resize_long_edge(image, max_size=900) that:
#   1. Reads the image's height and width from image.shape[:2].
#   2. Computes a scale factor as min(1.0, max_size / max(height, width)).
#   3. If scale == 1.0, returns image.copy() so small images stay untouched.
#   4. Otherwise, returns cv2.resize(image, (int(width * scale), int(height * scale))).


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--out", default="")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = cv2.imread(args.path)
    if image is None:
        raise SystemExit(f"Could not read image: {args.path}")

    print("shape:", image.shape)
    print("dtype:", image.dtype)
    print("min/max:", image.min(), image.max())
    print("note: OpenCV loads color images as BGR, not RGB")


if __name__ == "__main__":
    main()
```
Here is the complete code with the `resize_long_edge` helper function defined at the top of the file:

```python
import argparse
import cv2


# Define a helper function named resize_long_edge(image, max_size=900)
def resize_long_edge(image, max_size=900):
    # 1. Read height and width from image.shape[:2]
    height, width = image.shape[:2]

    # 2. Compute scale factor
    scale = min(1.0, max_size / max(height, width))

    # 3. If scale == 1.0, return a copy of the image
    if scale == 1.0:
        return image.copy()

    # 4. Otherwise, resize and return the downscaled image
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--out", default="")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = cv2.imread(args.path)
    if image is None:
        raise SystemExit(f"Could not read image: {args.path}")

    print("shape:", image.shape)
    print("dtype:", image.dtype)
    print("min/max:", image.min(), image.max())
    print("note: OpenCV loads color images as BGR, not RGB")


if __name__ == "__main__":
    main()

```

## Bringing the Preview Window to Life

With the resize_long_edge helper ready to go, it is time to wire up the last pieces of the workbench so your script can actually show and save images.

Inside main(), you will find four TODO comments waiting for you. Each one points to a single line you need to add to finish the tool.

Here is what to do:

    If args.out is a non-empty string, call cv2.imwrite(args.out, image) to save the full-resolution image to that path.
    Call cv2.imshow("preview", ...) and pass it the result of resize_long_edge(image, max_size=args.preview_size) so the preview window shows the resized image.
    Call cv2.waitKey(0) so the window stays open until a key is pressed.
    Call cv2.destroyAllWindows() to clean up the preview window afterward.

Once these lines are in place, your workbench will be a complete load-inspect-preview-save tool, ready to support the stitching work coming up next.

```
import argparse
import cv2


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--out", default="")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = cv2.imread(args.path)
    if image is None:
        raise SystemExit(f"Could not read image: {args.path}")

    print("shape:", image.shape)
    print("dtype:", image.dtype)
    print("min/max:", image.min(), image.max())
    print("note: OpenCV loads color images as BGR, not RGB")

    # TODO: If args.out is a non-empty string, call cv2.imwrite(args.out, image)
    #       to save a copy of the full-resolution image to that path.

    # TODO: Call cv2.imshow("preview", ...) passing the result of
    #       resize_long_edge(image, max_size=args.preview_size) so the
    #       preview window displays the resized image.

    # TODO: Call cv2.waitKey(0) to keep the preview window open until a key is pressed.

    # TODO: Call cv2.destroyAllWindows() to clean up the preview window once a key has been pressed.


if __name__ == "__main__":
    main()
```

Here is the complete implementation for `solution.py` with the image saving and preview window logic added:

```python
import argparse
import cv2


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--out", default="")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = cv2.imread(args.path)
    if image is None:
        raise SystemExit(f"Could not read image: {args.path}")

    print("shape:", image.shape)
    print("dtype:", image.dtype)
    print("min/max:", image.min(), image.max())
    print("note: OpenCV loads color images as BGR, not RGB")

    # If args.out is a non-empty string, save the full-resolution image to that path.
    if args.out:
        cv2.imwrite(args.out, image)

    # Display the resized image in a preview window named "preview".
    cv2.imshow("preview", resize_long_edge(image, max_size=args.preview_size))

    # Keep the preview window open until a key is pressed.
    cv2.waitKey(0)

    # Clean up and close the preview window.
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

## Testing the tool

Nice work assembling every piece of the workbench script across the last four exercises — now it's time to take it for a real spin.

The solution.py file is already complete and ready to use. In this exercise, your job is to run it from the terminal and confirm the full pipeline behaves the way you expect.

Try the following from the terminal:

    Run python solution.py sample_images/building/1.jpg and check that the printed shape, dtype, min/max, and BGR note all appear.
    Run it again with --preview-size 400 to see the preview window shrink.
    Add --out preview.png to save a copy of the loaded image to disk.
    Finally, try a bad path like python solution.py nope.jpg and confirm the script exits with the "Could not read image" message.

This is your first real tool in the stitching workbench — get comfortable with it, because every future exercise will build on top of it.

You're all set! Everything in your workbench pipeline is fully wired up and working.

Here is a quick summary of what happens with each command you run in the terminal:

1. **Standard Run:**
```bash
python solution.py sample_images/building/1.jpg

```


* **Result:** Loads the image, prints its metadata (`shape`, `dtype`, `min/max`, and the BGR warning), and displays the image fit within a default max dimension of 900px in the preview gallery.


2. **Custom Preview Size:**
```bash
python solution.py sample_images/building/1.jpg --preview-size 400

```


* **Result:** Re-calculates the scale factor using `400 / max(height, width)` and displays a smaller preview while keeping the original image array untouched in memory.


3. **Saving to Disk:**
```bash
python solution.py sample_images/building/1.jpg --out preview.png

```


* **Result:** Triggers `cv2.imwrite("preview.png", image)`, writing the original, unscaled full-resolution image array directly to `preview.png`.


4. **Error Handling Check:**
```bash
python solution.py nope.jpg

```


* **Result:** `cv2.imread()` returns `None`, safely catching the missing file and raising `SystemExit("Could not read image: nope.jpg")` without crashing OpenCV.



Your image workbench tool is now complete and ready to serve as the foundational step for feature detection and image stitching in the next lessons!